# 07 — Predict Face Age on a New Dataset

Apply the trained **Ridge morphometric model** (fitted on IXI) to any new collection of T1 MRI scans.

**Designed for SIMON** (73 sessions × 1 subject × 36 scanners) but works on any directory of `.nii.gz` or `.mgz` T1 files.

### What this notebook does
```
T1 scans (.nii.gz / .mgz)
    │
    ▼  extract_head_mesh()            isosurface + PyMeshLab cleanup
Head mesh (.ply)
    │
    ▼  center_and_extract_face()      crop + reorient to LSA space
Face mesh (.ply)
    │
    ▼  detect_landmarks()             BioFace3D-20 MVCNN  →  (20, 3) .npy
Landmarks
    │
    ├─▶  gpa_align_new()              project through IXI GPA — no refit
    │         mean_shape + PCA from data/features/gpa_mean_shape.npy / gpa_pca.joblib
    │
    └─▶  edma_form_matrix()           190 pairwise distances
    │
    ▼  StandardScaler.transform()     IXI train scaler — no refit
    │
    ▼  Ridge.predict()                best IXI benchmark model
    │
    ▼  results/simon_face_age.csv     session_id, face_age_pred, centroid_size, …
```

**Critical constraints:**
- The GPA and scaler are **never refit** on the new data — the IXI train-derived transforms are applied as-is.
- SIMON output is used for **scanner-variance analysis** (CV across 73 sessions), not MAE accuracy.


---
## 1. Configuration

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

# ── Paths ────────────────────────────────────────────────────────────────────
SIMON_DIR  = ROOT.parent / "data" / "simon"          # manifest.csv + images/
MESH_DIR   = ROOT / "results" / "simon_meshes"
OUT_DIR    = ROOT / "results" / "simon_face_age"

# IXI-derived artefacts (never refit on new data)
GPA_MEAN   = ROOT / "data" / "features" / "gpa_mean_shape.npy"
GPA_PCA    = ROOT / "data" / "features" / "gpa_pca.joblib"
SCALER     = ROOT / "data" / "models" / "benchmark" / "scaler.joblib"
MODEL      = ROOT / "data" / "models" / "benchmark" / "ridge_model.joblib"

# Landmark detection settings
DEVICE      = "mps"     # "mps" on Apple Silicon, "cuda" on GPU server, "cpu" fallback
PREDICT_NUM = 5         # MVCNN predictions to average per subject

MESH_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [GPA_MEAN, GPA_PCA, SCALER, MODEL]:
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {p.relative_to(ROOT)}")


---
## 2. Load Session Table

Reads `manifest.csv` and resolves local file paths under `data/simon/images/`.
Each row is one scan (session × run). Columns include `session_id`, `age`,
`manufacturer`, `scanner_model`, `institution`, `path`.

In [ ]:
import pandas as pd
from src.utils import load_simon_metadata

sessions = load_simon_metadata(SIMON_DIR)
print(f"Found {len(sessions)} scans")
sessions.head(10)


---
## 3. Stage 1 — Head Mesh → Face Mesh → Landmarks

Run extraction as a **script** (not in-process) to avoid Jupyter OOM kills.
Each subject's mesh pipeline (VTK isosurface + SimpleITK + PyMeshLab AO) uses
several hundred MB; running in a subprocess ensures memory is freed between subjects.

The script is fully checkpointed — safe to interrupt and re-run.
Already-completed subjects are skipped automatically.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(ROOT / "scripts" / "extract_simon_landmarks.py"),
    "--simon-dir", str(SIMON_DIR),
    "--device", DEVICE,
    "--predict-num", str(PREDICT_NUM),
]

# Run and stream output line by line so progress is visible in the notebook
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


---
## 4. Stage 2 — Features: GPA Projection + EDMA

The IXI GPA (mean shape + PCA) is **loaded from disk and applied without refitting**.
This ensures SIMON subjects are embedded in the same morphospace as the IXI training set.

- `gpa_align_new()` — translate, unit-scale, Procrustes-rotate to IXI mean shape, project through IXI PCA
- `edma_form_matrix()` — 190 upper-triangle pairwise distances (C(20,2))


In [ ]:
import joblib
from src.biomarkers import gpa_align_new, edma_form_matrix

# Load IXI-derived GPA artefacts
mean_shape = np.load(str(GPA_MEAN))          # (20, 3)
pca_model  = joblib.load(str(GPA_PCA))       # sklearn PCA fitted on IXI train (9 components)

_N_LM = 20
_TRIU_I, _TRIU_J = np.triu_indices(_N_LM, k=1)   # 190 pairs

def _edma_vec(lm: np.ndarray) -> np.ndarray:
    return edma_form_matrix(lm)[_TRIU_I, _TRIU_J].astype(np.float64)


# ── Collect landmarks and compute features ───────────────────────────────────
lm_list, valid_idx = [], []

for i, row in sessions.iterrows():
    sid = f"ses-{row['session_id']:03d}_run-{row['run']}"
    p = _lm_path(sid)
    if p.exists():
        lm_list.append(np.load(str(p)))
        valid_idx.append(i)

valid_df = sessions.loc[valid_idx].reset_index(drop=True)
print(f"Scans with landmarks: {len(lm_list)} / {len(sessions)}")

# GPA projection (no refit — IXI morphospace)
gpa_result     = gpa_align_new(lm_list, mean_shape, pca_model)
gpa_scores     = gpa_result["pc_scores"]             # (N, 9)
centroid_sizes = gpa_result["centroid_sizes"]         # (N,)
proc_dists     = gpa_result["procrustes_distances"]   # (N,)

# EDMA
edma_matrix = np.stack([_edma_vec(lm) for lm in lm_list])   # (N, 190)

# Combined feature matrix
X_raw = np.concatenate([gpa_scores, edma_matrix], axis=1)   # (N, 199)
print(f"Feature matrix: {X_raw.shape}  (gpa {gpa_scores.shape}, edma {edma_matrix.shape})")


---
## 5. Scale + Predict

The IXI train `StandardScaler` and the Ridge model are loaded from disk.
`scaler.transform()` is called — **not** `fit_transform()`.


In [ ]:
# Load IXI scaler and Ridge model
scaler = joblib.load(str(SCALER))
model  = joblib.load(str(MODEL))

# Scale (transform only — IXI train statistics)
X_scaled = scaler.transform(X_raw)

# Predict
y_pred = model.predict(X_scaled)

# ── Assemble full results table (all 99 sessions, NaN for failures) ───────────
results = sessions[["session_id", "run", "age", "manufacturer", "scanner_model", "institution"]].copy()
results["face_age_pred"]      = np.nan
results["error"]              = np.nan
results["abs_error"]          = np.nan
results["centroid_size"]      = np.nan
results["procrustes_dist"]    = np.nan
results["prediction_status"]  = "failed"

# Fill in values for sessions with successful landmark extraction
results.loc[valid_idx, "face_age_pred"]   = y_pred.round(2)
results.loc[valid_idx, "centroid_size"]   = centroid_sizes.round(4)
results.loc[valid_idx, "procrustes_dist"] = proc_dists.round(6)
results.loc[valid_idx, "prediction_status"] = "ok"

# Compute error only for successful predictions
mask = results["prediction_status"] == "ok"
results.loc[mask, "error"]     = (results.loc[mask, "face_age_pred"] - results.loc[mask, "age"]).round(2)
results.loc[mask, "abs_error"] = results.loc[mask, "error"].abs().round(2)

out_csv = OUT_DIR / "simon_face_age.csv"
results.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
print(f"  Total sessions: {len(results)}  |  successful: {mask.sum()}  |  failed: {(~mask).sum()}")
print(f"  MAE (successful): {results.loc[mask, 'abs_error'].mean():.2f} yr")
results.head(10)


---
## 6. Scanner-Variance Analysis

SIMON is a **single subject** scanned 73 times across 36 scanners.
The useful quantity here is not prediction accuracy (no ground-truth age per session)
but the **variability** of the predicted face-age across sessions — a proxy for how much
the morphometric signal is contaminated by scanner noise.

Compare directly with H3 from the README:
> H3: Face-age variability across scanners is lower than brain-age variability.


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

fa = results["face_age_pred"]
fa_mean = fa.mean()
fa_std  = fa.std()
fa_cv   = fa_std / fa_mean * 100

print(f"Face-age (morphometric Ridge):  {fa_mean:.1f} ± {fa_std:.1f} yr   CV = {fa_cv:.2f}%")
print(f"Range: {fa.min():.1f} – {fa.max():.1f} yr")
print(f"N sessions with predictions: {len(fa)}")


In [ ]:
# ── Session-order scatter ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(results["session_id"], results["face_age_pred"],
           alpha=0.7, color="steelblue", s=30, edgecolors="white", linewidths=0.4)
ax.axhline(fa_mean, color="black", lw=1.2, ls="--", label=f"mean = {fa_mean:.1f} yr")
ax.axhspan(fa_mean - fa_std, fa_mean + fa_std,
           alpha=0.12, color="steelblue", label=f"±1 SD ({fa_std:.1f} yr)")
ax.set_xlabel("Session index")
ax.set_ylabel("Predicted face-age (years)")
ax.set_title("Morphometric face-age across 73 SIMON sessions")
ax.legend(fontsize=9)

# ── Centroid size (head size proxy) vs face-age ───────────────────────────────
ax = axes[1]
sc = ax.scatter(results["centroid_size"], results["face_age_pred"],
                alpha=0.7, color="darkorange", s=30, edgecolors="white", linewidths=0.4)
if results["centroid_size"].std() > 0:
    r, p = pearsonr(results["centroid_size"], results["face_age_pred"])
    ax.set_title(f"Centroid size vs predicted face-age  (r={r:.2f}, p={p:.3f})")
else:
    ax.set_title("Centroid size vs predicted face-age")
ax.set_xlabel("Centroid size (Procrustes-normalised)")
ax.set_ylabel("Predicted face-age (years)")

plt.suptitle("SIMON — morphometric face-age variability (Ridge, IXI-trained)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "07_simon_variability.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Per-scanner breakdown ─────────────────────────────────────────────────────
scanner_stats = (
    results.groupby("scanner_model")["face_age_pred"]
    .agg(n="count", mean="mean", std="std")
    .assign(cv=lambda d: d["std"] / d["mean"] * 100)
    .round({"mean": 2, "std": 2, "cv": 2})
    .sort_values("mean")
)
print("Face-age by scanner model:")
display(scanner_stats)


---
## 7. Comparison with Brain-Age (optional)

If SynthSeg brain-age results for SIMON are available (from `02_simon_reliability.ipynb`),
load them here and compare CV values directly to test **H3**.


In [ ]:
SYNTHSEG_CSV = ROOT.parent / "results" / "simon_reliability" / "brain_ages_synthseg.csv"

if SYNTHSEG_CSV.exists():
    brain_df = pd.read_csv(SYNTHSEG_CSV)
    # Merge on session_id if possible, otherwise on filename stem
    if "session_id" in brain_df.columns:
        merged = results.merge(brain_df[["session_id", "brain_age"]], on="session_id", how="inner")
    else:
        brain_df["session_id"] = brain_df["filename"].str.extract(r"ses-(\d+)").astype(float)
        merged = results.merge(brain_df[["session_id", "brain_age"]], on="session_id", how="inner")

    fa_cv_m = merged["face_age_pred"].std() / merged["face_age_pred"].mean() * 100
    ba_cv_m = merged["brain_age"].std()     / merged["brain_age"].mean()     * 100

    print(f"Matched sessions: {len(merged)}")
    print(f"  Face-age  CV = {fa_cv_m:.2f}%  (std={merged['face_age_pred'].std():.2f} yr)")
    print(f"  Brain-age CV = {ba_cv_m:.2f}%  (std={merged['brain_age'].std():.2f} yr)")
    print()
    print("H3 (face-age CV < brain-age CV):", "SUPPORTED" if fa_cv_m < ba_cv_m else "NOT SUPPORTED")

    # Scatter face-age vs brain-age
    if merged["brain_age"].notna().sum() > 2:
        r, p = pearsonr(merged["face_age_pred"], merged["brain_age"])
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(merged["face_age_pred"], merged["brain_age"],
                   alpha=0.6, color="purple", s=30, edgecolors="white", linewidths=0.4)
        lims = [
            min(merged["face_age_pred"].min(), merged["brain_age"].min()) - 1,
            max(merged["face_age_pred"].max(), merged["brain_age"].max()) + 1,
        ]
        ax.plot(lims, lims, "k--", lw=0.8, label="identity")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("Morphometric face-age (Ridge)")
        ax.set_ylabel("Brain-age (SynthSeg)")
        ax.set_title(f"SIMON: face-age vs brain-age  (r={r:.2f}, p={p:.3f})")
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "07_simon_face_vs_brain_age.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print(f"SynthSeg results not found at:\n  {SYNTHSEG_CSV}")
    print("Run 02_simon_reliability.ipynb first, then re-run this cell.")


---
## 8. Save Full Feature Array (optional)

Saves a `.npz` in the same format as the IXI splits — useful for any follow-up analyses
that need direct access to GPA scores, EDMA distances, or centroid sizes.


In [ ]:
npz_path = OUT_DIR / "simon_features.npz"
np.savez_compressed(
    str(npz_path),
    session_ids          = valid_df["session_id"].values,
    runs                 = valid_df["run"].values,
    ages                 = valid_df["age"].values,
    face_age_pred        = y_pred,
    gpa_scores           = gpa_scores,
    edma_distances       = edma_matrix,
    centroid_sizes       = centroid_sizes,
    procrustes_distances = proc_dists,
)
print(f"Saved feature archive → {npz_path}")
print(f"  gpa_scores:    {gpa_scores.shape}")
print(f"  edma_distances:{edma_matrix.shape}")
